In [26]:
import os
import pydicom
import pandas as pd
from pathlib import Path

In [33]:
INPUT_ROOT = r"C:\Users\dosimetria4\Naiara_gdr\data\prove\Examples\FirstExample_471335\1"
OUTPUT_ROOT = r"C:\Users\dosimetria4\Naiara_gdr\data\prove\anonymized_Dataset\FirstExample_471335\1_anonymized"
MAPPING_FILE = r"C:\Users\dosimetria4\Naiara_gdr\data\prove\patients_mapping.csv"

MAPPING_FILE

'C:\\Users\\dosimetria4\\Naiara_gdr\\data\\prove\\patients_mapping.csv'

In [35]:
def load_mapping(mapping_path):
    print("Loading mapping file: {mapping_path}")

    df = pd.read_csv(mapping_path, dtype=str)
    df.columns = df.columns.str.strip()

    mapping_dict = pd.Series(df.ID.values, index = df.NHC.values).to_dict()

    return mapping_dict

In [ ]:
load_mapping(MAPPING_FILE)

Loading mapping file: {mapping_path}


{'289892': 'NCT07132658_1',
 '361840': 'NCT07132658_2',
 '394705': 'NCT07132658_3',
 '405913': 'NCT07132658_4',
 '416934': 'NCT07132658_5',
 '471335': 'NCT07132658_6',
 '499671': 'NCT07132658_7',
 '559465': 'NCT07132658_8',
 '630510': 'NCT07132658_9',
 '732566': 'NCT07132658_10',
 '983245': 'NCT07132658_11',
 '1067866': 'NCT07132658_12',
 '1074826': 'NCT07132658_13',
 '1081864': 'NCT07132658_14',
 '1083863': 'NCT07132658_15',
 '1087147': 'NCT07132658_16',
 '1100895': 'NCT07132658_17',
 '1102116': 'NCT07132658_18',
 '1105689': 'NCT07132658_19',
 '1108345': 'NCT07132658_20'}

In [30]:
def anonymize_dicom(dataset, new_id):
    """
    Applies NAIARA project specific anonymization rules.
    """
    # 1. Replace Identifiers
    dataset.PatientName = new_id
    dataset.PatientID = new_id

    # REMOVE BIRTH DATE (Anonymization)
    if 'PatientBirthDate' in dataset:
        dataset.PatientBirthDate = ""  # Clear value
    if 'PatientBirthTime' in dataset:
        dataset.PatientBirthTime = ""

    # 3. Clean other standard sensitive tags (Best Practice)
    # Remove references to hospital or physicians that might indirectly identify patient
    if 'ReferringPhysicianName' in dataset:
        dataset.ReferringPhysicianName = ""
    if 'OperatorsName' in dataset:
        dataset.OperatorsName = ""
        
    return dataset

In [57]:
def run_pipeline():
    # 1. Load mapping
    nhc_to_id = load_mapping(MAPPING_FILE)

    print(f"Starting scan of input directory: {INPUT_ROOT}")
    
    processed_count = 0
    unknown_count = 0
    error_count = 0

    # Walk through input directory
    for root, dirs, files in os.walk(INPUT_ROOT):
        for file in files:
            # Process only potential DICOM files
            if file.lower().endswith(".dcm") or "." not in file:
                file_path = os.path.join(root, file)
                try:
                    # Read DICOM file
                    # stop_before_pixels=False because we need to save the full file later
                    ds = pydicom.dcmread(file_path, force=True)
                    
                    # Extract NHC from DICOM
                    # NOTE: Usually in PatientID, sometimes in PatientName. Verify on site.
                    original_nhc = str(ds.PatientID).strip() 
                    
                    # Check if NHC exists in our mapping
                    if original_nhc in nhc_to_id:
                        new_anon_id = nhc_to_id[original_nhc]
                        
                        # Apply anonymization rules
                        ds = anonymize_dicom(ds, new_anon_id)
                        
                        # --- SAVING LOGIC ---
                        # Recreate folder structure under the NEW ID
                        # Example Output: OUTPUT_ROOT/NCT_ID/StudyDate/Series/.../file.dcm
                        
                        # Get relative path to keep series structure
                        rel_path = os.path.relpath(root, INPUT_ROOT)
                        
                        # Construct new path: Output -> New_ID -> Original Subfolders
                        # TODO: Missing the TREATMENT FOLDER
                        save_dir = os.path.join(OUTPUT_ROOT, rel_path)
                        
                        if not os.path.exists(save_dir):
                            os.makedirs(save_dir)
                            
                        save_path = os.path.join(save_dir, file)
                        ds.save_as(save_path)
                        processed_count += 1
                        
                    else:
                        # NHC found in DICOM but not in mapping file
                        unknown_count += 1
                        # Optional: Print only unique unknown NHCs to avoid spam
                        # print(f"⚠️ Unknown NHC: {original_nhc}")

                except Exception as e:
                    # Error reading file (not a DICOM or corrupted)
                    error_count += 1
                    # print(f"Error processing file {file}: {e}")
    
    print(f"\nPIPELINE FINISHED.")
    print(f"Successfully anonymized files: {processed_count}")
    print(f"Skipped files (NHC not in mapping): {unknown_count}")
    print(f"Read errors: {error_count}")

In [55]:
run_pipeline()

Loading mapping file: {mapping_path}
Starting scan of input directory: C:\Users\dosimetria4\Naiara_gdr\data\prove\Examples\FirstExample_471335\1

PIPELINE FINISHED.
Successfully anonymized files: 301
Skipped files (NHC not in mapping): 0
Read errors: 0
